# M1-INF4 — Python Workshop for Data Analysis
## Session 3 — Cleaning and validating data

A table can load successfully and still produce misleading results. Before analysing it, check what a row means, which values are valid, and which information is missing.

**Learning objectives**
- Audit missing values, data types and duplicate observations.
- Standardise text and convert numbers and dates while keeping track of failed conversions.
- Apply explicit validity rules and justify how missing values are handled.
- Build a reproducible cleaning pipeline, verify its result and export it.

You will reuse selection, Boolean masks, calculated columns, sorting and `groupby()` from the previous notebook.


### Working files
Place this notebook in `notebooks/`, next to the `data/` folder containing `temperatures.csv`. Run cells from top to bottom. Keep the original CSV unchanged.

The small training batch used below is embedded in the notebook; no additional download is required.


In [1]:
from pathlib import Path
import pandas as pd

# Works when the working directory is notebooks/ or the course root.
DATA_DIR = Path("../data") if Path("../data/temperatures.csv").exists() else Path("data")
DATA_FILE = DATA_DIR / "temperatures.csv"
if not DATA_FILE.exists():
    raise FileNotFoundError("Place temperatures.csv in the course data/ folder.")

# The first CSV column is an exported row index, not a measurement.
raw = pd.read_csv(DATA_FILE, index_col=0)
display(raw.head())


,date,city,country,avg_temp_c
0,2000-01-01,Abidjan,Côte D'Ivoire,27.293
1,2000-02-01,Abidjan,Côte D'Ivoire,27.685
2,2000-03-01,Abidjan,Côte D'Ivoire,29.061
3,2000-04-01,Abidjan,Côte D'Ivoire,28.162
4,2000-05-01,Abidjan,Côte D'Ivoire,27.547


## 1. Start with a data-quality audit

The table contains historical **monthly average temperatures**. One observation describes a city in a country for a month.

| Column | Meaning | Expected form |
|---|---|---|
| `date` | Month of observation | A valid date, recorded on the first day of the month |
| `city` | City name | Non-missing text |
| `country` | Country name | Non-missing text |
| `avg_temp_c` | Monthly average temperature | A number in degrees Celsius, or a missing value |

The proposed observation key is `country`, `city`, `date`. An index is a row label; it is not automatically a meaningful identifier.

A negative Celsius temperature is valid. A missing temperature is not zero.


In [2]:
print("Rows and columns:", raw.shape)
raw.info()
display(raw.describe(include="all"))


Rows and columns: (16500, 4)
<class 'pandas.DataFrame'>
RangeIndex: 16500 entries, 0 to 16499
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        16500 non-null  str    
 1   city        16500 non-null  str    
 2   country     16500 non-null  str    
 3   avg_temp_c  16407 non-null  float64
dtypes: float64(1), str(3)
memory usage: 515.8 KB


,date,city,country,avg_temp_c
count,16500,16500,16500,16407.000000
unique,165,100,49,NaN
top,2000-01-01,Abidjan,China,NaN
freq,100,165,2640,NaN
mean,NaN,NaN,NaN,19.732877
std,NaN,NaN,NaN,9.579635
min,NaN,NaN,NaN,-23.495000
25%,NaN,NaN,NaN,14.870000
50%,NaN,NaN,NaN,22.119000
75%,NaN,NaN,NaN,27.001500


### Counting missing values
`isna()` produces a Boolean table. For each column, `sum()` counts `True` values and `mean()` gives their proportion, because `True` behaves like 1 and `False` like 0 in these aggregations.


In [3]:
quality = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean() * 100,
    "dtype": raw.dtypes.astype(str)
})
display(quality.sort_values("missing_pct", ascending=False))


,missing_count,missing_pct,dtype
avg_temp_c,93,0.563636,float64
date,0,0.000000,str
city,0,0.000000,str
country,0,0.000000,str


### Your turn — What needs attention?
1. Display only rows with a missing temperature.
2. Count missing temperatures by city, in descending order. Hint: select the missing rows first, then use `groupby("city").size()`.
3. Calculate the percentage of missing temperatures in the whole table.
4. Explain why loading the file without an error does not mean the data is ready for analysis.


In [ ]:
#1 
raw.loc[raw["avg_temp_c"].isna()]
#Q2
raw.loc[raw["avg_temp_c"].isna()].groupby("city").size().sort_values(ascending=False)
#3  
#print((raw["avg_temp_c"].isna().sum() / len(raw)) * 100)

#4
# Le fichier peut se charger sans erreur tout en contenant des valeurs manquantes, des types incorrects, des doublons, etc. 
# Il est donc important de vérifier la qualité des données avant de les utiliser pour l'analyse.

0.5636363636363636


## 2. Standardise representations before making decisions

Work on a copy. Keep the raw values available so you can explain what changed.

### 2.1 Text labels
Whitespace and inconsistent case can split one category into several groups. `.str.strip()` removes leading and trailing whitespace; `.str.title()` standardises case for this exercise. In other datasets, acronyms and official spellings may require an explicit mapping instead.


In [ ]:
labels = pd.Series([" London ", "LONDON", "Paris", None], dtype="string")
display(pd.DataFrame({"before": labels, "after": labels.str.strip().str.title()}))


### 2.2 Numbers stored as text
`pd.to_numeric(..., errors="coerce")` converts values it can read and replaces invalid values with a missing value. It does not recover the correct measurement.

Here the input convention explicitly uses a comma as a decimal separator. Replace it with a dot before conversion; do not generalise this rule to thousands separators without checking the file specification.


In [ ]:
text_values = pd.Series(["12.5", "8,2", "unknown", None], dtype="string")
normalised_values = text_values.str.replace(",", ".", regex=False)
numeric_values = pd.to_numeric(normalised_values, errors="coerce")
failed_conversion = text_values.notna() & numeric_values.isna()
display(pd.DataFrame({
    "before": text_values,
    "after": numeric_values,
    "conversion_failed": failed_conversion
}))


### 2.3 Dates are more than text
`pd.to_datetime()` checks calendar validity. With `errors="coerce"`, an invalid date becomes `NaT`, the datetime missing value. Use the documented input format rather than guessing whether a date is day-first or month-first.


In [ ]:
date_text = pd.Series(["2001-01-01", "2001-02-30", None])
parsed_dates = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
display(pd.DataFrame({"before": date_text, "after": parsed_dates}))
display(parsed_dates.dt.year)


### Your turn — Convert, then explain
A sensor export contains the strings below. Convert the values to numbers and identify **newly created** missing values. Explain why counting all missing values after conversion would not give the same answer.

```python
sensor = pd.Series(["-2,5", "0", "missing", None], dtype="string")
```


In [1]:
# Your code here
import pandas as pd

# Série de départ
sensor = pd.Series(["-2,5", "0", "missing", None], dtype="string")

# A. Normaliser les virgules décimales en points
sensor_clean = sensor.str.replace(",", ".")

# B. Convertir en valeurs numériques (les valeurs invalides deviennent NaN/NaT)
sensor_num = pd.to_numeric(sensor_clean, errors="coerce")

# C. Identifier les NOUVELLES valeurs manquantes créées par la conversion
newly_created_missing = sensor.notna() & sensor_num.isna()

print("--- Série convertie en nombres ---")
print(sensor_num)

print("\n--- Nouvelles valeurs manquantes créées par la conversion ---")
print(sensor[newly_created_missing])

ImportError: DLL load failed while importing dtypes: Une stratégie de contrôle d’application a bloqué ce fichier.

## 3. Group problem 1 — Is this incoming batch ready?

The following batch is derived from a few London and Paris records. **Some values have deliberately been altered for this training exercise, and one row has been copied.** These issues must not be interpreted as defects in the original dataset.

Run the cell to create `incoming`. This is a separate object from `raw`.


In [3]:
incoming = pd.DataFrame(
[{'date': '2000-01-01',
  'city': ' london ',
  'country': 'United Kingdom',
  'avg_temp_c': 4.6930000000000005},
 {'date': '2000-02-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': '6,115'},
 {'date': '2000-03-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': 'not recorded'},
 {'date': '2000-02-30',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': 8.246},
 {'date': '2000-05-01',
  'city': 'London',
  'country': 'United Kingdom',
  'avg_temp_c': '999'},
 {'date': '2000-06-01',
  'city': 'London',
  'country': ' united kingdom ',
  'avg_temp_c': 15.875},
 {'date': '2000-01-01', 'city': ' PARIS ', 'country': 'France', 'avg_temp_c': 3.845},
 {'date': '2000-02-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': None},
 {'date': '2000-03-01',
  'city': 'Paris',
  'country': 'FRANCE ',
  'avg_temp_c': 7.872000000000001},
 {'date': '2000-04-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 10.067},
 {'date': '2000-05-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 15.451},
 {'date': '2000-06-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 17.666},
 {'date': '2000-04-01', 'city': 'Paris', 'country': 'France', 'avg_temp_c': 10.067}]
)
display(incoming)


NameError: name 'pd' is not defined

### Batch specification
- City and country labels must be stripped of surrounding whitespace and use title case.
- Dates use `YYYY-MM-DD` and must be valid first days of a month.
- Temperatures use Celsius. A comma may be a decimal separator.
- For **this exercise**, temperatures outside **−50 to 60 °C** require review; this is a screening rule, not a universal scientific validity range.
- There should be at most one observation per country, city and month.

### Your investigation
1. Preserve `incoming` and create `prepared = incoming.copy()`.
2. Audit missing values, types and exact duplicate rows **before** changing anything.
3. Standardise city/country text and convert temperatures and dates.
4. Display the original rows whose non-missing temperatures or dates could not be converted.
5. Display numeric temperatures outside the screening range.
6. Propose a policy for each issue: correct a representation, retain a missing value, remove a confirmed duplicate, or set a row aside for review. **Do not delete rows yet.**

**Group deliverable:** code, a small issue report, and a short explanation of your proposed policy. Each member must be able to explain the masks used.


### Your group answer


In [4]:
# 1. Copie pour ne pas altérer l'original
prepared = incoming.copy()

# 2. Audit initial avant modifications
print("=== 2. AUDIT INITIAL ===")
print("Valeurs manquantes (isna) :\n", prepared.isna().sum())
print("\nTypes de données :\n", prepared.dtypes)
print("\nLignes strictement doublonnées :", prepared.duplicated().sum())

# 3. Standardisation des textes, dates et températures
prepared['city'] = prepared['city'].astype(str).str.strip().str.title()
prepared['country'] = prepared['country'].astype(str).str.strip().str.title()

# Conversion des dates (errors='coerce' transforme les dates invalides comme 2000-02-30 en NaT)
dates_converted = pd.to_datetime(prepared['date'], format='%Y-%m-%d', errors='coerce')

# Conversion des températures (remplacement virgule + coercion)
temp_clean = prepared['avg_temp_c'].astype(str).str.replace(',', '.')
temps_converted = pd.to_numeric(temp_clean, errors='coerce')

# 4. Identification des échecs de conversion (masques)
invalid_date_mask = prepared['date'].notna() & dates_converted.isna()
invalid_temp_mask = prepared['avg_temp_c'].notna() & temps_converted.isna()

print("\n=== 4. LIGNES AVEC DATES INVALIDES ===")
print(incoming[invalid_date_mask])

print("\n=== 4. LIGNES AVEC TEMPÉRATURES NON CONVERTIBLES ===")
print(incoming[invalid_temp_mask])

# Appliquer les séries converties dans le dataframe préparé
prepared['date'] = dates_converted
prepared['avg_temp_c'] = temps_converted

# 5. Détection des températures hors intervalle [-50, 60] °C
out_of_bounds_mask = prepared['avg_temp_c'].notna() & ((prepared['avg_temp_c'] < -50) | (prepared['avg_temp_c'] > 60))

print("\n=== 5. TEMPÉRATURES HORS INTERVALLE (-50 à 60 °C) ===")
print(incoming[out_of_bounds_mask])

NameError: name 'incoming' is not defined

### Issue Report & Proposed Policy

#### Liste des incidents identifiés

* **Ligne 1 (London / UK - `avg_temp_c = '6,115'`) :**
  * **Problème :** Présence d'une virgule comme séparateur décimal.
  * **Décision :** Correction automatique en remplaçant la virgule par un point (`6.115`) et conversion en type numérique.

* **Ligne 2 (London / UK - `avg_temp_c = 'not recorded'`) :**
  * **Problème :** Texte explicite indiquant une absence de mesure.
  * **Décision :** Conversion en valeur manquante (`NaN`) et conservation de la ligne pour l'analyse.

* **Ligne 3 (London / UK - `date = '2000-02-30'`) :**
  * **Problème :** Date impossible (le 30 février n'existe pas).
  * **Décision :** Mise de côté pour révision. Nécessite une confirmation externe de la date réelle.

* **Ligne 4 (London / UK - `avg_temp_c = '999'`) :**
  * **Problème :** Température aberrante hors de la plage définie ($-50$ à $60\ ^\circ\text{C}$).
  * **Décision :** Mise de côté pour révision (code d'erreur probable du capteur).

* **Ligne 7 (Paris / France - `avg_temp_c = None`) :**
  * **Problème :** Donnée manquante d'origine.
  * **Décision :** Conservation sous forme de `NaN`.

* **Ligne 12 (Paris / France - `2000-04-01`) :**
  * **Problème :** Doublon parfait de la ligne 9.
  * **Décision :** Suppression du doublon (`drop_duplicates()`).

---

### Synthèse des politiques appliquées

1. **Standardisation du texte :** Suppression des espaces superflus et passage au format Title Case (`.str.strip().str.title()`) sur les colonnes `city` et `country`.
2. **Normalisation numérique :** Remplacement des virgules par des points pour la conversion en flottants.
3. **Gestion des données manquantes :** Les valeurs absentes (`None`) ou textuelles (`'not recorded'`) sont homogénéisées en `NaN`.
4. **Mise de côté pour audit :** Isolation des incohérences calendaires et des valeurs aberrantes sans modifier la donnée brute originale.

## 4. Duplicates: identical rows or conflicting observations?

`duplicated()` compares whole rows by default. `duplicated(subset=key, keep=False)` identifies **all** rows sharing an observation key. A repeated key may contain conflicting measurements, so it is not automatically safe to keep the first row.


In [ ]:
example = pd.DataFrame({
    "city": ["London", "London", "Paris", "Paris"],
    "date": ["2000-01-01"] * 4,
    "avg_temp_c": [4.7, 4.7, 5.0, 7.0]
})
print("Extra identical rows:", example.duplicated().sum())
unique_rows = example.drop_duplicates().copy()
conflicts = unique_rows.duplicated(subset=["city", "date"], keep=False)
display(unique_rows.loc[conflicts])


The London rows are identical; if confirmed as a repeated import, keep one. The Paris values disagree for the same key. Set them aside for review instead of silently choosing a temperature. In the supplied monthly data, use `country`, `city`, `date` together as the key.


## 5. Missing values: choose a policy that fits the question

- Missing observation keys prevent identification: set these rows aside for review.
- Missing measurements can be retained and counted. pandas `mean()` skips missing measurements by default.
- `dropna(subset=[...])` removes rows missing the listed fields. Plain `dropna()` may remove more rows than intended.
- `fillna(0)` asserts a value of zero. For temperature, zero is a real measurement, not a neutral placeholder.

A result should report both the statistic and how much data supports it.


In [ ]:
measurements = pd.Series([10.0, 20.0, float("nan")])
print("Mean of available measurements:", measurements.mean())
print("Mean after inventing a zero:", measurements.fillna(0).mean())
print("Available / total:", measurements.count(), "/", measurements.size)


### An explicit policy for our monthly table
Keep missing temperatures in the cleaned table, and report their absence in summaries. Set invalid-key rows and out-of-range temperatures aside for review. Keep one copy of a confirmed exact duplicate. Set conflicting keys aside rather than choosing arbitrarily.

An empty label should also count as missing after trimming. A temperature that fails numeric conversion becomes missing, but the original value and failure count remain in the audit.


In [ ]:
# Small illustration: these labels all mean an absent identifier.
empty_labels = pd.Series(["Paris", "   ", None], dtype="string")
display(empty_labels.str.strip().replace("", pd.NA))


### Toolkit — Combining checks and validating results

- `table[key].isna().any(axis=1)` flags a row if **any** key field is missing. `axis=1` means “across columns, for each row”.
- `table["date"].dt.day` extracts the day of a parsed date; `.ne(1)` means “not equal to 1”.
- `values.between(-50, 60)` includes both endpoints. Check `notna()` separately to distinguish missing values from out-of-range numbers.
- `table.loc[mask].copy()` creates a separate table for flagged rows. Assign a string column such as `review["reason"] = "invalid date"` to document a decision.
- In a grouped summary, `size` counts rows and `count` counts non-missing values in the selected column.
- `assert condition` stops execution if a promised property is false. For example, `assert len(before) == len(kept) + len(rejected)` checks row accounting when no duplicate copies have been removed. Include those copies in the accounting when necessary.

Checks support a documented policy; they do not prove that every retained measurement is correct.


## 6. Group problem 2 — Deliver an auditable clean table

Build a reproducible pipeline starting again from `incoming.copy()`. Do not rely on someone having completed problem 1 correctly.

Apply the policy above:
1. Standardise labels; treat empty labels as missing. Convert numbers and dates.
2. Remove confirmed exact duplicate rows after standardisation. Record how many copies were removed.
3. Set aside rows with a missing country/city/date, a date that is not the first of its month, or a non-missing temperature outside −50 to 60 °C.
4. Among the remaining rows, set aside **all** rows with conflicting observation keys.
5. Keep the other rows, including missing temperatures, as `clean_batch`. Keep rejected records in `review_batch` and add a reason column.
6. Produce a per-city summary with total rows, available temperatures and mean temperature. Explain the difference between `size` and `count`.
7. Verify that every input row is accounted for: input rows = clean rows + review rows + removed duplicate copies.
8. Save the clean table and review table as CSV files with `index=False`.

**Group deliverable:** reproducible code, both tables, the row-accounting check, and three sentences explaining your decisions and their impact on the mean. Rotate who writes and who reviews the code.

The export filenames are `temperatures_batch_clean.csv` and `temperatures_batch_review.csv`. Keep them in a `cleaned/` folder next to this notebook.


### Your group answer


In [5]:
import os
import pandas as pd

# 0. Réinitialisation à partir de incoming
prepared = incoming.copy()

# 1. Standardisation des libellés, dates et températures
prepared['city'] = prepared['city'].astype(str).str.strip().str.title()
prepared['country'] = prepared['country'].astype(str).str.strip().str.title()

# Remplacement des chaînes vides par NA
prepared['city'] = prepared['city'].replace('', pd.NA)
prepared['country'] = prepared['country'].replace('', pd.NA)

# Conversions
prepared['date_parsed'] = pd.to_datetime(prepared['date'], format='%Y-%m-%d', errors='coerce')

temp_clean = prepared['avg_temp_c'].astype(str).str.replace(',', '.')
prepared['temp_parsed'] = pd.to_numeric(temp_clean, errors='coerce')

# 2. Suppression des doublons exacts après standardisation
# On reconstruit un dataframe temporaire standardisé pour identifier les doublons stricts
df_std = pd.DataFrame({
    'date': prepared['date_parsed'],
    'city': prepared['city'],
    'country': prepared['country'],
    'avg_temp_c': prepared['temp_parsed']
})

dup_mask = df_std.duplicated(keep='first')
removed_duplicates_count = dup_mask.sum()

# Conservation des lignes non-doublons
df = prepared[~dup_mask].copy()

# 3 & 4. Détection des raisons de rejet
reasons = []

for idx, row in df.iterrows():
    row_reasons = []
    
    # Manque country / city / date
    if pd.isna(row['country']) or pd.isna(row['city']) or pd.isna(row['date_parsed']):
        row_reasons.append("Missing country/city/date")
    # Date non premier du mois
    elif row['date_parsed'].day != 1:
        row_reasons.append("Date is not the 1st of the month")
        
    # Température hors intervalle [-50, 60] °C (seulement si non manquante)
    if pd.notna(row['temp_parsed']) and (row['temp_parsed'] < -50 or row['temp_parsed'] > 60):
        row_reasons.append("Temperature outside -50 to 60 °C")
        
    reasons.append("; ".join(row_reasons) if row_reasons else None)

df['reason'] = reasons

# Détection des clés en conflit (clé d'observation = country + city + date_parsed)
valid_dates_mask = df['date_parsed'].notna()
key_counts = df[valid_dates_mask].groupby(['country', 'city', 'date_parsed'])['date'].transform('count')

# Marquer les doublons de clés
conflict_mask = valid_dates_mask & (key_counts > 1)
df.loc[conflict_mask, 'reason'] = df.loc[conflict_mask, 'reason'].apply(
    lambda r: (r + "; Conflicting observation key") if r else "Conflicting observation key"
)

# 5. Séparation en clean_batch et review_batch
review_mask = df['reason'].notna()

review_batch = df[review_mask].copy()
review_batch['date'] = review_batch['date_parsed']
review_batch['avg_temp_c'] = review_batch['temp_parsed']
review_batch = review_batch[['date', 'city', 'country', 'avg_temp_c', 'reason']]

clean_batch = df[~review_mask].copy()
clean_batch['date'] = clean_batch['date_parsed']
clean_batch['avg_temp_c'] = clean_batch['temp_parsed']
clean_batch = clean_batch[['date', 'city', 'country', 'avg_temp_c']]

# 6. Résumé par ville (per-city summary)
city_summary = clean_batch.groupby('city')['avg_temp_c'].agg(
    total_rows='size',
    available_temperatures='count',
    mean_temperature='mean'
).reset_index()

print("=== RESUME PAR VILLE ===")
print(city_summary)

# 7. Vérification du bilan des lignes (Row-accounting check)
n_input = len(incoming)
n_clean = len(clean_batch)
n_review = len(review_batch)
n_duplicates = removed_duplicates_count

print("\n=== VERIFICATION DES LIGNES ===")
print(f"Lignes en entrée : {n_input}")
print(f"Lignes propres   : {n_clean}")
print(f"Lignes à revoir  : {n_review}")
print(f"Doublons retirés : {n_duplicates}")
print(f"Total vérifié    : {n_clean + n_review + n_duplicates} (Égalité : {n_input == n_clean + n_review + n_duplicates})")

# 8. Sauvegarde des fichiers CSV
os.makedirs("cleaned", exist_ok=True)
clean_batch.to_csv("cleaned/temperatures_batch_clean.csv", index=False)
review_batch.to_csv("cleaned/temperatures_batch_review.csv", index=False)
print("\nFichiers sauvegardés avec succès dans le dossier 'cleaned/'.")

ImportError: DLL load failed while importing dtypes: Une stratégie de contrôle d’application a bloqué ce fichier.

##### Explication de la différence entre `size` et `count`
* **`size`** compte le nombre total de lignes (observations) attribuées à un groupe, qu'il y ait des valeurs manquantes (`NaN`) ou non.
* **`count`** compte uniquement le nombre de valeurs non manquantes (valides) dans la colonne ciblée (`avg_temp_c`).

---

##### Explication des décisions et impact sur la moyenne (3 phrases)

1. **Isolation des aberrations et erreurs de date :** La valeur extrême de 999 °C et la date invalide du 30 février ont été écartées vers le panier de révision (`review_batch`) afin d'éviter la corruption des séries temporelles.
2. **Conservation des valeurs manquantes légitimes :** Les relevés absents (comme `None` ou `'not recorded'`) ont été conservés dans le jeu de données propre sous forme de `NaN` pour préserver le nombre réel d'observations attendues par ville.
3. **Impact sur la moyenne :** L'exclusion de la valeur aberrante de 999 °C empêche un biais massif à la hausse de la moyenne de Londres, garantissant ainsi un calcul physiquement cohérent sur les données conservées.

## 7. Transfer to the complete dataset

Return to the original `raw` table. The training issues were deliberately introduced; do not assume they occur in the full CSV.

Your aim is to prepare the table for exploratory analysis while documenting its remaining limitations.


### Your turn — Ready for exploration?
1. Starting from `raw.copy()`, convert dates and check the monthly observation key, exact duplicates and the screening range.
2. Retain missing temperatures. Confirm how many are missing, without replacing them with zero.
3. For each country and city, report total observations, available temperatures, mean temperature and missing percentage.
4. Export `temperatures_clean.csv` with `index=False` into `cleaned/`.
5. Explain why cities with different time coverage should not be compared using their overall means alone.

You may reuse your pipeline from problem 2. If you did not finish it, write these steps directly for the full table.


In [ ]:
import os
import pandas as pd

# 1. Copie à partir de raw
df = raw.copy()

# Standardisation des textes
df['city'] = df['city'].astype(str).str.strip().str.title()
df['country'] = df['country'].astype(str).str.strip().str.title()
df['city'] = df['city'].replace('', pd.NA)
df['country'] = df['country'].replace('', pd.NA)

# Conversion des dates et des températures
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')

temp_clean = df['avg_temp_c'].astype(str).str.replace(',', '.')
df['avg_temp_c'] = pd.to_numeric(temp_clean, errors='coerce')

# Vérification des doublons stricts
exact_duplicates_count = df.duplicated(keep='first').sum()
df = df.drop_duplicates(keep='first').copy()

# Filtrage : élimination des dates invalides / non 1er du mois / températures hors intervalle
valid_rows_mask = (
    df['country'].notna() &
    df['city'].notna() &
    df['date'].notna() &
    (df['date'].dt.day == 1) &
    (df['avg_temp_c'].isna() | ((df['avg_temp_c'] >= -50) & (df['avg_temp_c'] <= 60)))
)

clean_data = df[valid_rows_mask].copy()

# Vérification des doublons de clé mensuelle (country, city, date)
key_duplicates_count = clean_data.duplicated(subset=['country', 'city', 'date'], keep=False).sum()
clean_data = clean_data.drop_duplicates(subset=['country', 'city', 'date'], keep='first').copy()

# 2. Décompte des températures manquantes (sans les remplacer par zéro)
missing_temps_count = clean_data['avg_temp_c'].isna().sum()
print(f"Nombre de températures manquantes conservées (NaN) : {missing_temps_count}")

# 3. Rapport par pays et par ville
summary_report = clean_data.groupby(['country', 'city'])['avg_temp_c'].agg(
    total_observations='size',
    available_temperatures='count',
    mean_temperature='mean'
).reset_index()

# Calcul du pourcentage de valeurs manquantes
summary_report['missing_percentage'] = (
    (summary_report['total_observations'] - summary_report['available_temperatures']) 
    / summary_report['total_observations'] * 100
)

print("\n=== RAPPORT PAR PAYS ET VILLE ===")
print(summary_report)

# 4. Export de temperatures_clean.csv dans le dossier cleaned/
os.makedirs("cleaned", exist_ok=True)
clean_data.to_csv("cleaned/temperatures_clean.csv", index=False)
print("\nFichier 'cleaned/temperatures_clean.csv' sauvegardé avec succès.")

###### Comparaison des moyennes globales entre villes

Il ne faut pas comparer la température moyenne globale de deux villes ayant des couvertures temporelles différentes pour deux raisons principales :

1. **Biais de saisonnalité :** Si une ville dispose de données principalement enregistrées en été et une autre en hiver, la moyenne globale de la première sera artificiellement surestimée.
2. **Incohérence des périodes d'observation :** Les tendances climatiques à long terme ou les anomalies météorologiques d'une année spécifique faussent la comparaison si les plages d'années ne se chevauchent pas exactement.

**Conclusion :** Pour effectuer une comparaison valide, il convient de comparer des périodes identiques (par exemple uniquement sur les mêmes mois et les mêmes années) ou de calculer des moyennes désaisonnalisées.

### Check the exported file
CSV stores values and column names, but does not preserve all pandas types. Parse dates again when loading it. `index=False` prevents an unintended extra index column.


In [ ]:
# Run after completing the export above.
OUTPUT_DIR = Path("cleaned")
export_file = OUTPUT_DIR / "temperatures_clean.csv"
if export_file.exists():
    reloaded = pd.read_csv(export_file, parse_dates=["date"])
    display(reloaded.head())
    reloaded.info()
else:
    print("Complete the export exercise to create temperatures_clean.csv.")


## 8. OPTIONAL / IF TIME ALLOWS — Categories and memory

Repeated labels can be stored with the `category` dtype. This changes storage and type semantics; it does not clean invalid labels or make a number meaningful.

A numeric identifier such as a store code is not automatically a quantity. Similarly, do not convert arbitrary text such as `"False"` to Boolean with `astype(bool)`: non-empty strings are truthy. Use an explicit mapping when values encode yes/no.


In [ ]:
memory_demo = raw.copy()
before = memory_demo.memory_usage(deep=True).sum()
for col in ["city", "country"]:
    memory_demo[col] = memory_demo[col].astype("category")
after = memory_demo.memory_usage(deep=True).sum()
print("Before (bytes):", before)
print("After (bytes):", after)
print("Reduction (%):", round((before - after) / before * 100, 1))


## Key takeaways

- Diagnose problems before dropping rows or filling values.
- Distinguish a representation fix from an invented measurement.
- Preserve raw data, record exclusions and count every row.
- Check both identical rows and conflicts on the observation key.
- Validate the cleaned table against explicit rules.
- Report missingness and coverage alongside statistics.

**Next:** explore distributions, compare groups and visualise data, using the quality checks to interpret what the figures can and cannot show.
